# Task 7: ResNet-50 Style Residual Bottleneck Block

### Goal
Build a simple custom ResNet-50 style bottleneck block using:
- PyTorch
- TorchVision
- PyTorch Profiler

In [ ]:
import torch
import torch.nn as nn
import torchvision

from torch.profiler import profile, ProfilerActivity

## 1. Custom ResNet-50 Style Bottleneck



In [ ]:
class CustomBottleneck(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()

        # Reduce the number of channels.
        mid_channels = out_channels // 4

        # 1x1 convolution for channel reduction.
        self.conv1 = nn.Conv2d(
            in_channels,
            mid_channels,
            kernel_size=1,
            bias=False
        )

        self.bn1 = nn.BatchNorm2d(mid_channels)

        # Depthwise 3x3 convolution.
        # Each channel has its own spatial filter.
        self.depthwise = nn.Conv2d(
            mid_channels,
            mid_channels,
            kernel_size=3,
            padding=1,
            groups=mid_channels,
            bias=False
        )

        self.bn2 = nn.BatchNorm2d(mid_channels)

        # 1x1 convolution expands the channels again.
        self.conv3 = nn.Conv2d(
            mid_channels,
            out_channels,
            kernel_size=1,
            bias=False
        )

        self.bn3 = nn.BatchNorm2d(out_channels)

        # Learnable skip projection.
        # It is needed when input and output channels differ.
        if in_channels != out_channels:
            self.skip = nn.Sequential(
                nn.Conv2d(
                    in_channels,
                    out_channels,
                    kernel_size=1,
                    bias=False
                ),
                nn.BatchNorm2d(out_channels)
            )
        else:
            self.skip = nn.Identity()

        self.relu = nn.ReLU()


    def forward(self, x):

        identity = self.skip(x)

        out = self.relu(self.bn1(self.conv1(x)))

        out = self.relu(self.bn2(self.depthwise(out)))

        out = self.bn3(self.conv3(out))

        # Residual connection
        out = out + identity

        return self.relu(out)

## 2. Test the Custom Block

In [ ]:
# Example input
x = torch.randn(1, 64, 56, 56)

block = CustomBottleneck(
    in_channels=64,
    out_channels=256
)

output = block(x)

print("Input shape :", x.shape)
print("Output shape:", output.shape)

## 3. Count Trainable Parameters

In [ ]:
def count_parameters(model):
    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )


print(
    "Trainable parameters:",
    count_parameters(block)
)

## 4. Run PyTorch Profiler



In [ ]:
with profile(
    activities=[ProfilerActivity.CPU],
    record_shapes=True,
    profile_memory=True,
    with_flops=True
) as prof:

    for _ in range(5):
        output = block(x)

print(
    prof.key_averages().table(
        sort_by="self_cpu_time_total",
        row_limit=10
    )
)

## 5. Get Total FLOPs

The profiler may not report FLOPs for every operation. We add the reported convolution FLOPs here.

In [ ]:
total_flops = 0

for event in prof.key_averages():
    if event.flops is not None:
        total_flops += event.flops

print("Estimated FLOPs:", total_flops)

## 6. Compare Channel Scaling Factors



In [ ]:
scales = [128, 256, 512]

results = []

for channels in scales:

    model = CustomBottleneck(
        in_channels=64,
        out_channels=channels
    )

    test_input = torch.randn(1, 64, 56, 56)

    with profile(
        activities=[ProfilerActivity.CPU],
        profile_memory=True,
        with_flops=True
    ) as p:

        model(test_input)

    flops = 0

    for event in p.key_averages():
        if event.flops is not None:
            flops += event.flops

    parameters = count_parameters(model)

    results.append(
        [channels, parameters, flops]
    )


print("Channels | Parameters | FLOPs")
print("-" * 40)

for row in results:
    print(
        f"{row[0]:8d} | "
        f"{row[1]:10d} | "
        f"{row[2]:.0f}"
    )

## 7. Compare With TorchVision ResNet-50

TorchVision already provides a standard ResNet-50 model. We only inspect its parameter count here as a reference.

The custom block above is **not** intended to reproduce the complete ResNet-50 network.

In [ ]:
resnet50 = torchvision.models.resnet50(weights=None)

print(
    "TorchVision ResNet-50 parameters:",
    count_parameters(resnet50)
)